# Building the JAX MAOOAM tendency with `dqgs`

This notebook shows how to use `src/dqgs/tendency_py2jax.py` to get a pure-JAX,
differentiable tendency `dU/dt = f(U)` for the fixed 36-D qgs MAOOAM
configuration.

It covers:

1. Enabling float64 and importing the module.
2. Evaluating the full tendency `tendency(U, params)`.
3. Building a parameter-recovery interface with `make_tendency` so that only a
   chosen handful of physical parameters are exposed to autodiff.
4. Taking gradients / Jacobians w.r.t. the state and w.r.t. the free parameters.

The module is standalone (it does **not** import qgs); it is a verbatim
transcription of the symbolic tendency exported by
`configs/create_tendencies_nonfixed_params.py`.


## 1. Enable float64, then import

MAOOAM needs double precision, so `jax_enable_x64` **must** be set before any
JAX arrays are created. `dqgs` deliberately does not flip this flag for you;
`require_x64()` asserts it is on.


In [3]:
import jax
jax.config.update("jax_enable_x64", True)  # must come before any jax array work

import jax.numpy as jnp
import numpy as np

import dqgs
from dqgs import (
    tendency,
    make_tendency,
    PARAMETER_NAMES,
    DEFAULT_PARAMS,
    DEFAULT_PARAM_VECTOR,
    STATE_DIM,
    require_x64,
)

require_x64()  # raises if x64 is off
print("state dim:", STATE_DIM)
print("n free params:", len(PARAMETER_NAMES))

state dim: 36
n free params: 16


## 2. The free parameters and their defaults

`PARAMETER_NAMES` is the 16-vector signature order; `DEFAULT_PARAMS` holds the
standard-config values (already nondimensionalized by qgs where relevant — do
not hand-edit them). `DEFAULT_PARAM_VECTOR` is the same values as a float64
array, ordered to match `PARAMETER_NAMES`.


In [2]:
for name in PARAMETER_NAMES:
    print(f"{name:>8s} = {DEFAULT_PARAMS[name]}")

     k_d = 0.029
     k_p = 0.029
   sigma = 0.2
     g_p = 0.031
       r = 0.0009689922480620154
       h = 136.5
       d = 0.001065891472868217
 gamma_a = 10000000.0
    C_a1 = 103.3333
 epsilon = 0.7
    T_a0 = 289.3
      sc = 1.0
    lmda = 15.06
 gamma_o = 560000000.0
   C_go1 = 310.0
   T_go0 = 301.46


## 3. Evaluate the full tendency

`tendency(U, params)` maps a 36-D state and the full 16-D parameter vector to
`dU/dt`. Here we use a small random state and the default parameters.


In [14]:
decruz_ic = np.load("../configs/ic/de_cruz_active_ic.npy")
U = decruz_ic

In [ ]:
dUdt = tendency(decruz_ic, DEFAULT_PARAM_VECTOR)
print("dU/dt shape:", dUdt.shape, "dtype:", dUdt.dtype)
print(dUdt)

dU/dt shape: (36,) dtype: float64
[-8.41125076e-05 -2.62579705e-03 -6.41636485e-04 -2.70868540e-04
 -1.08023227e-03  2.97051669e-03 -1.69382413e-03 -2.40437711e-03
  5.09185227e-04  1.13406813e-03 -5.31479892e-04 -1.14872735e-03
 -1.69176053e-04 -3.47992196e-04  7.62592394e-04  1.03413650e-03
 -7.09029786e-04 -2.08094828e-04  1.44074936e-04  1.61464502e-04
 -3.48755059e-09 -3.52230036e-09 -8.90804501e-11  3.46315002e-10
 -1.28186448e-08 -6.36682416e-10 -4.01559123e-11 -1.61881160e-10
  3.06144534e-06 -1.74017441e-06 -2.82789126e-06 -1.22973594e-06
 -7.55121722e-06  6.53411043e-06  1.56734347e-07 -6.62375619e-08]


## 4. The parameter-recovery interface: `make_tendency`

For inverse problems we usually want to differentiate w.r.t. just a few physical
parameters while holding the rest fixed. `make_tendency(free_names)` returns a
function `f_theta(U, theta)` where `theta` carries only the chosen parameters
(ordered as `free_names`); everything else is baked in from the defaults.

This means `jax.grad` / `jax.jacfwd` w.r.t. `theta` differentiate **only** those
1-3 parameters — exactly the recovery use case.


In [15]:
free_names = ["k_d", "sigma", "lmda"]
f_theta = make_tendency(free_names)

theta = jnp.array([DEFAULT_PARAMS[n] for n in free_names])
print("theta (free params):", dict(zip(free_names, theta.tolist())))

# Sanity check: with theta = defaults, f_theta matches the full tendency.
err = jnp.max(jnp.abs(f_theta(U, theta) - tendency(U, DEFAULT_PARAM_VECTOR)))
print("max |f_theta - tendency| at defaults:", float(err))

theta (free params): {'k_d': 0.029, 'sigma': 0.2, 'lmda': 15.06}
max |f_theta - tendency| at defaults: 0.0


## 5. Differentiate w.r.t. the state and the parameters

Because everything is pure JAX, the usual transforms compose. Two common
quantities:

- **State Jacobian** `df/dU` (shape 36x36) — needed for tangent-linear / adjoint
  integration and stability analysis.
- **Parameter Jacobian** `df/dtheta` (shape 36 x n_free) — the sensitivity of the
  tendency to the parameters you want to recover.


In [5]:
# State Jacobian of the full tendency, df/dU at fixed params.
J_state = jax.jacfwd(tendency, argnums=0)(U, DEFAULT_PARAM_VECTOR)
print("df/dU shape:", J_state.shape)

# Parameter Jacobian of the free-parameter tendency, df/dtheta.
J_theta = jax.jacfwd(f_theta, argnums=1)(U, theta)
print("df/dtheta shape:", J_theta.shape, "(36 x %d free params)" % len(free_names))

df/dU shape: (36, 36)


df/dtheta shape: (36, 3) (36 x 3 free params)


### Gradient of a scalar loss w.r.t. the free parameters

A toy parameter-recovery objective: integrate one Euler step and compare to a
target. `jax.grad` w.r.t. `theta` gives the search direction over only the free
parameters.


In [6]:
dt = 0.01
U_target = U + dt * tendency(U, DEFAULT_PARAM_VECTOR)  # "data" from true params

def loss(theta):
    U_next = U + dt * f_theta(U, theta)
    return jnp.sum((U_next - U_target) ** 2)

# At the true params the loss and gradient are ~0.
g = jax.grad(loss)(theta)
print("loss at true theta:", float(loss(theta)))
print("grad at true theta:", g)

# Perturb one parameter and confirm a nonzero gradient points back.
theta_off = theta.at[0].multiply(1.1)  # bump k_d by 10%
print("loss at perturbed theta:", float(loss(theta_off)))
print("grad at perturbed theta:", jax.grad(loss)(theta_off))

loss at true theta: 0.0
grad at true theta: [0. 0. 0.]
loss at perturbed theta: 6.754096826276913e-13
grad at perturbed theta: [4.65799781e-10 7.71684594e-12 8.90301671e-14]


## Summary

- Set `jax_enable_x64` **before** importing / using anything.
- `tendency(U, params)` is the full 36-D, 16-parameter right-hand side.
- `make_tendency(free_names)` bakes in the fixed parameters and exposes only the
  ones you want to recover as a small `theta` — then `jax.grad` / `jax.jacfwd`
  give clean state- and parameter-derivatives for assimilation / recovery.

Next steps (not in this notebook): an RK4 + `lax.scan` integrator for full
trajectories, and validating against qgs (see `docs/agent/validation.md`).
